## Cell 1: Your Turn - Paste My Rule

My Rule:
The search results should show the most relevant and helpful items at the top.

Why this matters:
If bad/low-quality items rank higher, users get frustrated and leave.

Expected Action Codes - Top 10 things I will check/fix:
1.  R01: Title not matching what the user searched for
2.  R02: Too short or missing description  
3.  R03: Item has low engagement - no clicks, no views
4.  R04: Item is old/outdated but ranking high
5.  R05: Duplicate items showing up multiple times
6.  R06: Spam or low-quality content ranking high
7.  R07: Missing important tagsor keywords
8.  R08: New Orleans good items are buried too far down
9.  R09: Popular items not getting boosted enough
10. R10: Wrong language/location items showing first

# Cell 2: Load Data + Build Ranked Queue

In [24]:
# 2. BUILD THE RANKED QUEUE
import pandas as pd
import numpy as np
from google.colab import files

print("Upload content_refresh_anonymized.csv")
uploaded = files.upload()  # a box will pop up to upload your file

df = pd.read_csv('content_refresh_anonymized.csv')
print(f"Loaded {len(df)} rows")

# Example baseline: rank by a simple signal.
# Change this to match your lane - for Ranking Signals we use ctr, traffic_drop, or priority
if 'priority_score' in df.columns:
    df['rank'] = df['priority_score'].rank(ascending=False)
elif 'ctr' in df.columns:
    df['rank'] = df['ctr'].rank(ascending=False)
else:
    df['rank'] = np.random.rand(len(df)) # fallback if no signal column

top_100 = df.sort_values('rank').head(100)
print("Top 100 queue built")
top_100.head()

Upload content_refresh_anonymized.csv


Saving content_refresh_anonymized.csv to content_refresh_anonymized (2).csv
Loaded 30000 rows
Top 100 queue built


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,rank
22607,content_bc2c0c7243df,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,873.0,6239.0,...,100.0,1.0,0.0,0.0,0.0,low,top_3,flat,NaN,6.0
21147,content_6016b918a48f,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,766.0,5837.0,...,100.0,30.0,0.0,0.0,0.0,low,page_3_5,flat,NaN,6.0
19598,content_a84e013a5f94,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,936.0,6558.0,...,100.0,9.0,0.0,0.0,0.0,low,page_1,flat,NaN,6.0
18825,content_98458bafe297,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,1316.0,9110.0,...,100.0,1.0,0.0,0.0,0.0,low,top_3,flat,NaN,6.0
6473,content_cfa4d9f1bf0a,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,2972.0,21585.0,...,100.0,25.0,0.0,0.0,0.0,low,page_3_5,new,NaN,6.0


# Cell 3: Get Baseline Score

In [25]:
# 3. BASELINE SCORE - Using ctr
import pandas as pd

# Clean ctr column
top_100['ctr'] = pd.to_numeric(top_100['ctr'], errors='coerce')
top_100 = top_100.dropna(subset=['ctr'])

# Baseline: % of items in top 20% ctr
threshold = top_100['ctr'].quantile(0.8)
high_value_count = (top_100['ctr'] >= threshold).sum()
baseline_score = (high_value_count / len(top_100)) * 100

print(f"Total items scored: {len(top_100)}")
print(f"Using signal: ctr")
print(f"Using threshold: ctr >= {threshold:.4f}")
print(f"BASELINE SCORE: {baseline_score:.2f}/100")
print("Screenshot this number for Week 4 submission")

Total items scored: 100
Using signal: ctr
Using threshold: ctr >= 50.0000
BASELINE SCORE: 44.00/100
Screenshot this number for Week 4 submission


In [26]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'rank']


In [27]:
print("Your columns are:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

Your columns are:
1. content_id
2. client_id
3. search_volume
4. competition
5. competition_level
6. cpc
7. content_type
8. main_intent
9. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct
45. rank


# Cell 4: The Rule + Flag

In [28]:
# 4. RULE + REASON CODE + ACTION
def apply_rule(row):
    if row['ctr'] >= 50:
        return 100, 'high_ctr', 'keep_ranking'
    elif row['ctr'] >= 10:
        return 50, 'mid_ctr', 'test_title'
    else:
        return 0, 'low_ctr', 'optimize_meta'

top_100[['score', 'reason_code', 'action']] = top_100.apply(apply_rule, axis=1, result_type='expand')
top_100.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,rank,score,reason_code,action
22607,content_bc2c0c7243df,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,873.0,6239.0,...,0.0,0.0,low,top_3,flat,NaN,6.0,100,high_ctr,keep_ranking
21147,content_6016b918a48f,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,766.0,5837.0,...,0.0,0.0,low,page_3_5,flat,NaN,6.0,100,high_ctr,keep_ranking
19598,content_a84e013a5f94,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,936.0,6558.0,...,0.0,0.0,low,page_1,flat,NaN,6.0,100,high_ctr,keep_ranking
18825,content_98458bafe297,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,1316.0,9110.0,...,0.0,0.0,low,top_3,flat,NaN,6.0,100,high_ctr,keep_ranking
6473,content_cfa4d9f1bf0a,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,2972.0,21585.0,...,0.0,0.0,low,page_3_5,new,NaN,6.0,100,high_ctr,keep_ranking
25755,content_a8cee66e4788,client_d4735e3a26,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,0.0,0.0,low,top_3,down,-100.0,6.0,100,high_ctr,keep_ranking
7514,content_b1e4f7904d85,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,791.0,5588.0,...,0.0,0.0,low,top_3,flat,NaN,6.0,100,high_ctr,keep_ranking
240,content_006b16e7a2e7,client_9f14025af0,0.0,0.0,LOW,0.0,keyword article,informational,2522.0,17424.0,...,50.0,0.0,low,top_3,flat,NaN,6.0,100,high_ctr,keep_ranking
19341,content_4272d3a330a3,client_9f14025af0,0.0,0.0,LOW,0.0,keyword article,informational,3005.0,20441.0,...,25.0,0.0,low,page_1,flat,NaN,6.0,100,high_ctr,keep_ranking
13661,content_bf398aa7400e,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,816.0,6204.0,...,0.0,0.0,low,top_3,flat,NaN,6.0,100,high_ctr,keep_ranking


# Cell 5: Save Ranked Queue CSV

In [33]:
# 5. WRITE RANKED QUEUE - Fixed version
import os

# 1. Create the folder if it doesn't exist
os.makedirs('work/outputs', exist_ok=True)

# 2. Save the CSV
top_100.sort_values('score', ascending=False).to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Saved to work/outputs/baseline_action_score.csv")

Saved to work/outputs/baseline_action_score.csv


In [30]:
print(top_100.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'rank', 'score', 'reason_code', 'action']


# Cell 5.5

In [32]:
import os
os.makedirs('work/outputs', exist_ok=True)
print("Folder created")

Folder created


# Cell 6: Top-10 Review

In [45]:
# 6. TOP-10 REVIEW - MATCHES YOUR ACTUAL CSV
import pandas as pd

df = pd.read_csv('work/outputs/baseline_action_score.csv')

# Convert to numbers
for col in ['ctr', 'clicks_90d', 'impressions_90d', 'rank', 'score', 'avg_position']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

top10 = df.sort_values('score', ascending=False).head(10)

for i, row in top10.iterrows():
    print(f"{i}. Content ID: {row['content_id']}")
    print(f" Score: {row['score']:.1f} | CTR: {row['ctr']:.4f} | Rank: {row['rank']:.0f}")
    print(f" Clicks_90d: {int(row['clicks_90d'])} | Impressions_90d: {int(row['impressions_90d'])}")
    print(f" Action: {row['action']} | Reason: {row['reason_code']}")
    print("---")

0. Content ID: content_bc2c0c7243df
 Score: 100.0 | CTR: 100.0000 | Rank: 6
 Clicks_90d: 1 | Impressions_90d: 1
 Action: keep_ranking | Reason: high_ctr
---
1. Content ID: content_6016b918a48f
 Score: 100.0 | CTR: 100.0000 | Rank: 6
 Clicks_90d: 1 | Impressions_90d: 1
 Action: keep_ranking | Reason: high_ctr
---
2. Content ID: content_a84e013a5f94
 Score: 100.0 | CTR: 100.0000 | Rank: 6
 Clicks_90d: 1 | Impressions_90d: 1
 Action: keep_ranking | Reason: high_ctr
---
3. Content ID: content_98458bafe297
 Score: 100.0 | CTR: 100.0000 | Rank: 6
 Clicks_90d: 1 | Impressions_90d: 1
 Action: keep_ranking | Reason: high_ctr
---
4. Content ID: content_cfa4d9f1bf0a
 Score: 100.0 | CTR: 100.0000 | Rank: 6
 Clicks_90d: 1 | Impressions_90d: 1
 Action: keep_ranking | Reason: high_ctr
---
5. Content ID: content_a8cee66e4788
 Score: 100.0 | CTR: 100.0000 | Rank: 6
 Clicks_90d: 1 | Impressions_90d: 1
 Action: keep_ranking | Reason: high_ctr
---
6. Content ID: content_b1e4f7904d85
 Score: 100.0 | CTR: 1

In [35]:
df = pd.read_csv('work/outputs/baseline_action_score.csv')
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'rank', 'score', 'reason_code', 'action']


In [38]:
import pandas as pd
df = pd.read_csv('work/outputs/baseline_action_score.csv')
print("Columns:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'rank', 'score', 'reason_code', 'action']

First 3 rows:
             content_id          client_id  search_volume  competition  \
0  content_bc2c0c7243df  client_d4735e3a26            NaN          NaN   
1  content_6016

In [44]:
import pandas as pd
df = pd.read_csv('work/outputs/baseline_action_score.csv')
print("ALL COLUMNS:", list(df.columns))
print("\nFirst row:")
print(df.iloc[0])

ALL COLUMNS: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'rank', 'score', 'reason_code', 'action']

First row:
content_id                content_bc2c0c7243df
client_id                    client_d4735e3a26
search_volume                              NaN
competition          